In [ ]:
import pandas as pd

In [ ]:
# Load raw business case dataset
df = pd.read_csv("business_case_dataset_CSV.csv")
df.head()

In [ ]:
df.info()

### Extracting country and regions

In [ ]:
!pip install pycountry

In [ ]:
import pycountry

def extract_country(description):
    for country in pycountry.countries:
        if country.name in description:
            return country.name
    return "Unknown"

df["Country"] = df["Contract Description"].apply(extract_country) 

In [ ]:
df['Country'].value_counts()

In [ ]:
country_replacements = {
    "United States": ["USA", "U.S.", "United States of America"],
    "United Kingdom": ["UK", "Britain", "England"],
    "Democratic Republic of the Congo": ["DR Congo", "D.R. Congo", "Congo-Kinshasa"],
    "Republic of Congo": ["Congo-Brazzaville"],
    "South Korea": ["Korea Republic"],
    "North Korea": ["Korea DPR"],
    "Russia": ["Russian Federation"],
}

def enhanced_extract_country(description):
    for country, alt_names in country_replacements.items():
        if any(alt in description for alt in alt_names):
            return country
    for country in pycountry.countries:
        if country.name in description:
            return country.name
    return "Unknown"

df["Country"] = df["Contract Description"].apply(enhanced_extract_country)

In [ ]:
df.to_csv('business_case_data_with_country.csv', index = False)

A step is missing here after the previous cell, which maps countries to their regions and sub-regions. The intermediate dataset called 'Dataset_with_UN_Regions.csv' is then imported below to continue the data preparation

### Adding Proposal Status

In [ ]:
# Load raw business case dataset
df = pd.read_csv("Dataset_with_UN_Regions.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df["Proposal Status"] = "Won"

In [ ]:
# Creating synthetic lost proposals 

import numpy as np

# Sample 50% of the data for synthetic lost cases
lost_cases = df.sample(frac=0.5, random_state=42).copy()
lost_cases["Proposal Status"] = "Lost"

# Modify some fields
lost_cases["Contract Award Amount (USD)"] *= np.random.uniform(0.8, 1.2, size=len(lost_cases))
lost_cases["Award Date"] = pd.to_datetime(lost_cases["Award Date"]) + pd.to_timedelta(np.random.randint(5, 60, size=len(lost_cases)), unit='D')

# Combine
df_full = pd.concat([df, lost_cases], ignore_index=True)
df_full.head(10)

In [ ]:
df_full['Proposal Status'].value_counts()

In [ ]:
# Save for modeling
df_full.to_csv("prepared_rfp_dataset.csv", index=False)